In [ ]:
import os
# Prevent JAX from pre-allocating all GPU memory (needed for working in multi-user/VC environments)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["JAX_DISABLE_MMAP_CACHE"] = "1"
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=2"

import glob, time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
jax.clear_caches()
import jax.numpy as jnp

import discovery as ds
from enterprise_extensions import load_feathers
from discovery.deterministic import make_phase_connected_binary
from discovery import const as disco_const
from discovery.deterministic import fpcmu_fast

# Enterprise imports for simulation
import scipy.linalg as sl
import scipy.sparse as ss
from sksparse.cholmod import cholesky
from enterprise.signals import signal_base
from enterprise.signals.gp_signals import get_timing_model_basis, BasisGP
from enterprise.signals.parameter import function
from enterprise.signals import white_signals, gp_signals, utils, selections, parameter
from enterprise_extensions.blocks import common_red_noise_block
from enterprise_extensions.deterministic import cw_block_circ

print('Imports OK')

In [ ]:
# =============================================================================
# CONFIGURABLE PARAMETERS — CHANGE THESE

In [ ]:
# =============================================================================
Npulsars = 5        # Number of pulsars to include (max 116 available)
log10_h  = -12.0    # log10(strain amplitude). -12 = high SNR, -13 = moderate

# Toggle: whether to sample red noise / GWB hyperparameters or fix at injection
# False = fix red noise & GWB at injected values (simpler, same CW+dist parameter space)
# True  = also sample rednoise_log10_A, rednoise_gamma per pulsar + gwb_log10_A, gwb_gamma
SAMPLE_NOISE = False

# Sampler schedule: 3 phases
n_anneal = 15000    # Phase 1: annealing steps (T: T_start -> 1)
n_adapt  = 5000     # Phase 2: adaptive covariance burn-in at T=1
n_prod   = 5000     # Phase 3: production sampling at T=1
T_start  = 5000.0   # Starting temperature (higher = broader initial exploration)
T_end    = 1.0      # Final temperature (1.0 = sample the true posterior)

# Noise injection parameters
INJECT_REDNOISE = True   # Whether to inject intrinsic red noise per pulsar
INJECT_GWB      = False  # Whether to inject gravitational wave background

KPC_OVER_C = disco_const.kpc / disco_const.c  # kpc -> light-seconds conversion

print(f"Configuration: N_CW=1, Npulsars={Npulsars}, h=10^{log10_h}")
print(f"SAMPLE_NOISE={SAMPLE_NOISE}, INJECT_REDNOISE={INJECT_REDNOISE}, INJECT_GWB={INJECT_GWB}")

feather_dir = "../data_products/"

In [ ]:
# =============================================================================
# LOAD PULSARS — both discovery and enterprise versions

In [ ]:
# =============================================================================
disco_psrs = [ds.Pulsar.read_feather(f) for f in sorted(glob.glob(feather_dir + "*.feather"))][:Npulsars]
print(f"Loaded {len(disco_psrs)} pulsars: {[p.name for p in disco_psrs]}")

# Enterprise pulsars (needed for simulate() and distance priors)
psrs_ent = load_feathers.load_feathers_from_folder(feather_dir)
ent_by_name = {p.name: p for p in psrs_ent}
# Keep only the pulsars we're using, in the same order
psrs_ent_used = [ent_by_name[p.name] for p in disco_psrs]

# Load EM distance priors
dist_mu, dist_sig = [], []
for psr in disco_psrs:
    ep  = ent_by_name[psr.name]
    mu  = float(ep.pdist[0])
    sig = float(ep.pdist[1]) if len(ep.pdist) > 1 else 0.5
    if (not np.isfinite(sig)) or sig <= 0:
        sig = 0.5
    dist_mu.append(mu)
    dist_sig.append(sig)

dist_mu_jnp = jnp.array(dist_mu, dtype=jnp.float64)
sd_jnp      = jnp.array(dist_sig, dtype=jnp.float64)
mu_arr = np.array(dist_mu)
sd_arr = np.array(dist_sig)

# Pre-extract per-pulsar data
psr_toas_list = [np.asarray(psr.toas, dtype=np.float64) for psr in disco_psrs]
psr_pos_list  = [psr.pos for psr in disco_psrs]
psr_positions = jnp.array([psr.pos for psr in disco_psrs])

print(f"dist_mu:  {[f'{m:.3f}' for m in dist_mu]}")
print(f"dist_sig: {[f'{s:.4f}' for s in dist_sig]}")

In [ ]:
# =============================================================================
# INJECTION PARAMETERS

In [ ]:
# =============================================================================
CW_PARAM_NAMES = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc', 'log10_fgw', 'log10_h', 'phase0', 'psi']

INJ = {
    "cos_gwtheta": 0.3,  "gwphi":    2.5,  "cos_inc":  -0.2,
    "log10_mc":    9.0,  "log10_fgw": -8.0, "log10_h": log10_h,
    "phase0":      1.0,  "psi":       0.7,
}

cw_func = make_phase_connected_binary(pulsarterm=True)

# True distances: offset from EM prior by 0.3 sigma
DIST_OFFSET_SIGMA = 0.3
true_dists = {psr.name: float(dist_mu[i]) + DIST_OFFSET_SIGMA * float(dist_sig[i])
              for i, psr in enumerate(disco_psrs)}

In [ ]:
# =============================================================================
# ENTERPRISE PTA MODEL — for data simulation via simulate()

In [ ]:
# =============================================================================
# Enterprise simulate() does a Cholesky decomposition of the full noise covariance
# (white noise + red noise + GWB) and draws a noise realization from it.
# This is the standard way to generate realistic PTA data.

# The simulate() function from the sandbox notebook
def simulate(pta, params, sparse_cholesky=True):
    """
    Simulate fake PTA residuals using enterprise's simulate().
    For each pulsar:
      1. Compute deterministic delays (e.g., CW signal)
      2. Draw a red noise + GWB realization from the full GP covariance
         (joint Fourier basis, Cholesky of phi matrix)
      3. Draw white noise
    Returns delay + GP_noise + white_noise per pulsar.
    """
    delays, ndiags, fmats, phis = (pta.get_delay(params=params),
                                   pta.get_ndiag(params=params),
                                   pta.get_basis(params=params),
                                   pta.get_phi(params=params))

    gpresiduals = []
    if pta._commonsignals:
        if sparse_cholesky:
            cf = cholesky(ss.csc_matrix(phis))
            gp = np.zeros(phis.shape[0])
            gp[cf.P()] = np.dot(cf.L().toarray(), np.random.randn(phis.shape[0]))
        else:
            gp = np.dot(sl.cholesky(phis, lower=True), np.random.randn(phis.shape[0]))

        i = 0
        for fmat in fmats:
            j = i + fmat.shape[1]
            gpresiduals.append(np.dot(fmat, gp[i:j]))
            i = j
        assert len(gp) == i
    else:
        for fmat, phi in zip(fmats, phis):
            if phi is None:
                gpresiduals.append(0)
            elif phi.ndim == 1:
                gpresiduals.append(np.dot(fmat, np.sqrt(phi) * np.random.randn(phi.shape[0])))
            else:
                raise NotImplementedError

    whiteresiduals = []
    for delay, ndiag in zip(delays, ndiags):
        if ndiag is None:
            whiteresiduals.append(0)
        elif isinstance(ndiag, signal_base.ShermanMorrison):
            n = np.diag(ndiag._nvec)
            for j, s in zip(ndiag._jvec, ndiag._slices):
                n[s, s] += j
            whiteresiduals.append(delay + np.dot(sl.cholesky(n, lower=True), np.random.randn(n.shape[0])))
        elif ndiag.ndim == 1:
            whiteresiduals.append(delay + np.sqrt(ndiag) * np.random.randn(ndiag.shape[0]))
        else:
            raise NotImplementedError

    return [np.array(g + w) for g, w in zip(gpresiduals, whiteresiduals)]


@function
def tm_prior(weights, toas, variance=1e40):
    return weights * variance * len(toas)

def TimingModel(coefficients=False, name="linear_timing_model",
                use_svd=False, normed=True, prior_variance=1e40):
    # Enterprise timing model with improper prior — needed for simulate() to
    # include timing model basis in the GP covariance. The variance=1e40 is
    # intentionally huge (improper prior).
    basis = get_timing_model_basis(use_svd, normed)
    prior = tm_prior(variance=prior_variance)
    BaseClass = BasisGP(prior, basis, coefficients=coefficients, name=name)
    class TimingModel(BaseClass):
        signal_type = "basis"
        signal_name = "linear timing model"
        signal_id = name + "_svd" if use_svd else name
    return TimingModel

print("Building enterprise PTA model for data simulation...")

# Zero out residuals before building the model
for psr in psrs_ent_used:
    psr._pdist = psr.pdist
    psr.residuals = np.array(psr.toas) * 0.0

tmin = [p.toas.min() for p in psrs_ent_used]
tmax = [p.toas.max() for p in psrs_ent_used]
Tspan = np.max(tmax) - np.min(tmin)

# White noise: fixed efac=1, equad=-8 (matching the sandbox)
selection = selections.Selection(selections.by_backend)
efac  = parameter.Constant(1)
equad = parameter.Constant(-8)

# Red noise priors (needed for model structure even when fixing params)
log10_A_red = parameter.Uniform(-18, -11)
gamma_red   = parameter.Uniform(0, 7)

# GWB priors
log10_A_gw = parameter.Uniform(-18, -11)('gwb_log10_A')
gamma_gw   = parameter.Uniform(0, 7)('gwb_gamma')

components = 30  # Number of Fourier components

tm = TimingModel(coefficients=False, name="linear_timing_model",
                 use_svd=False, normed=True, prior_variance=1e-14)

models = []
for p in psrs_ent_used:
    s = tm
    ef = white_signals.MeasurementNoise(efac=efac, selection=selection)
    s += ef
    eq = white_signals.TNEquadNoise(log10_tnequad=equad, selection=selection)
    s += eq

    pl = utils.powerlaw(log10_A=log10_A_red, gamma=gamma_red)
    rn = gp_signals.FourierBasisGP(spectrum=pl, components=components, Tspan=Tspan, name="rednoise")
    s += rn

    crn = common_red_noise_block(psd='powerlaw', prior='log-uniform',
                                  components=components, orf='hd', name='gwb')
    s += crn

    # CW source (phase-connected, with pulsar term)
    s += cw_block_circ(
        amp_prior="log-uniform", dist_prior=None, skyloc=None,
        log10_fgw=None, psrTerm=True, phase_connected=True,
        discoclone=False, name="cw",
    )
    models.append(s(p))

pta = signal_base.PTA(models)
pta.set_default_params({})
print(f"Enterprise PTA model built with {len(pta.pulsars)} pulsars")

In [ ]:
# =============================================================================
# BUILD ENTERPRISE PARAMETER DICTIONARY FOR SIMULATION

In [ ]:
# =============================================================================
np.random.seed(42)  # Reproducible noise injection

enterprise_params = {}

# Red noise: draw from realistic ranges (or set to negligible if not injecting)
if INJECT_REDNOISE:
    for p in pta.param_names:
        if 'rednoise_log10_A' in p:
            enterprise_params[p] = np.random.uniform(-16, -13)
        elif 'rednoise_gamma' in p:
            enterprise_params[p] = np.random.uniform(2, 6)
else:
    # Set red noise amplitude to negligible level
    for p in pta.param_names:
        if 'rednoise_log10_A' in p:
            enterprise_params[p] = -18.0
        elif 'rednoise_gamma' in p:
            enterprise_params[p] = 3.0

# GWB
if INJECT_GWB:
    enterprise_params['gwb_gamma']   = 4.333
    enterprise_params['gwb_log10_A'] = -14.5
else:
    enterprise_params['gwb_gamma']   = 4.333
    enterprise_params['gwb_log10_A'] = -18.0

# Pulsar distances
for psr in psrs_ent_used:
    for p in pta.param_names:
        if psr.name + '_cw_p_dist' in p:
            enterprise_params[p] = true_dists[psr.name]

# CW parameters
enterprise_params.update({
    "cw_cos_gwtheta": INJ["cos_gwtheta"],
    "cw_gwphi":       INJ["gwphi"],
    "cw_cos_inc":     INJ["cos_inc"],
    "cw_log10_mc":    INJ["log10_mc"],
    "cw_log10_fgw":   INJ["log10_fgw"],
    "cw_log10_h":     INJ["log10_h"],
    "cw_phase0":      INJ["phase0"],
    "cw_psi":         INJ["psi"],
})

print("\nInjection parameters:")
for k, v in sorted(enterprise_params.items()):
    print(f"  {k}: {v}")

In [ ]:
# =============================================================================
# SIMULATE DATA — includes CW signal + white noise + red noise + GWB

In [ ]:
# =============================================================================
print("\nSimulating data with enterprise...")
sim_resids = simulate(pta, enterprise_params, sparse_cholesky=True)
name_to_resid = {getattr(p, "name", p): y for p, y in zip(pta.pulsars, sim_resids)}

# Map to discovery pulsar order
data_list = [name_to_resid[psr.name] for psr in disco_psrs]
print(f"Simulated residuals: {[f'{np.std(d):.2e}' for d in data_list]}")

In [ ]:
# =============================================================================
# DISCOVERY LIKELIHOOD SETUP — PulsarLikelihood + ArrayLikelihood

In [ ]:
# =============================================================================
# This replaces the hand-rolled chi-squared likelihood with discovery's
# Woodbury-marginalized likelihood that properly handles:
#   - Heteroscedastic white noise (efac, equad per backend)
#   - Timing model marginalization (improper prior GP)
#   - Intrinsic red noise per pulsar (Fourier basis GP)
#   - HD-correlated GWB (global Fourier basis GP)
#   - CW signal as a deterministic delay
print("\nBuilding discovery likelihood...")

# Build noisedict with fixed white noise parameters (efac=1, log10_t2equad=-8)
# These match what was used in the simulation
noisedict = {}
noisedict.update({psr.name + "_KAT_MKBF_efac": 1.0 for psr in disco_psrs})
noisedict.update({psr.name + "_KAT_MKBF_log10_t2equad": -8.0 for psr in disco_psrs})

# Noise terms: fixed white noise per pulsar (no free parameters since noisedict is complete)
noise_terms = {psr.name: ds.makenoise_measurement(psr, noisedict=noisedict) for psr in disco_psrs}

# Timing model: improper-prior GP that marginalizes over the timing model
timing_terms = {psr.name: ds.makegp_timing(psr, variance=1e-14) for psr in disco_psrs}

# Tspan for Fourier basis
T_disco = ds.getspan(disco_psrs)

# Red noise: per-pulsar Fourier GP (common structure, independent amplitudes)
common_gp = ds.makecommongp_fourier(disco_psrs, ds.powerlaw, 30, T_disco, name="rednoise")

# GWB: Hellings-Downs correlated global GP
global_gp = ds.makeglobalgp_fourier(disco_psrs, ds.powerlaw, ds.hd_orf, 30, T_disco, name="gwb")


# CW delay callable — adapts make_phase_connected_binary to discovery's dict interface
class SingleSourceDelay:
    """Wraps the CW delay function for a single pulsar to accept a params dict.
    
    Discovery's PulsarLikelihood expects delay functions to be callables that take
    a params dict and return a time-domain delay vector. This class adapts the
    make_phase_connected_binary function to that interface.
    """
    def __init__(self, psr):
        self.psr = psr
        self.cw = make_phase_connected_binary(pulsarterm=True)
        # These are the parameter names that discovery will look for in the params dict
        self.params = [
            "cw_cos_gwtheta", "cw_gwphi", "cw_cos_inc", "cw_log10_mc",
            "cw_log10_fgw", "cw_log10_h", "cw_phase0", "cw_psi",
            f"{psr.name}_cw_p_dist",
        ]  # .params tells discovery which dict keys this delay depends on, so it can trace parameter dependencies.

    def __call__(self, params):
        return self.cw(
            self.psr.toas, self.psr.pos,
            cos_gwtheta=params["cw_cos_gwtheta"],
            gwphi=params["cw_gwphi"],
            cos_inc=params["cw_cos_inc"],
            log10_mc=params["cw_log10_mc"],
            log10_fgw=params["cw_log10_fgw"],
            log10_h=params["cw_log10_h"],
            phase0=params["cw_phase0"],
            psi=params["cw_psi"],
            p_dist=params[f"{self.psr.name}_cw_p_dist"],
        )


# Build PulsarLikelihood for each pulsar
pulsar_likes = [
    ds.PulsarLikelihood([
        np.array(data_list[i], copy=True),  # observed residuals
        noise_terms[psr.name],               # white noise kernel
        timing_terms[psr.name],              # timing model GP
        SingleSourceDelay(psr),              # CW delay function
    ])
    for i, psr in enumerate(disco_psrs)
]

# ArrayLikelihood combines all pulsars with common/global GPs
fml = ds.ArrayLikelihood(pulsar_likes, commongp=common_gp, globalgp=global_gp)
logl_disco = fml.logL  # callable: params dict -> scalar log-likelihood

print(f"Discovery likelihood built. Parameters: {logl_disco.params}")

In [ ]:
# =============================================================================
# PARAMETER MAPPING — flat vector <-> dict

In [ ]:
# =============================================================================
# The sampler works with a flat numpy array. We need to map between this and
# the discovery params dict. The layout depends on SAMPLE_NOISE.

# Bookkeeping: map enterprise param names to discovery names for the injection dict
# Discovery uses "cw_" prefix; enterprise also uses "cw_" for our single-source case
disco_inj_params = {}
for k, v in enterprise_params.items():
    disco_inj_params[k] = v

# The parameters that logl_disco expects, partitioned:
# 1. CW params (always sampled): cw_cos_gwtheta, cw_gwphi, etc.
# 2. Distance params (always sampled): {psr.name}_cw_p_dist
# 3. Red noise params (sampled if SAMPLE_NOISE): {psr.name}_rednoise_log10_A, {psr.name}_rednoise_gamma
# 4. GWB params (sampled if SAMPLE_NOISE): gwb_log10_A, gwb_gamma

# CW param names in discovery's convention
CW_DISCO_NAMES = [
    "cw_cos_gwtheta", "cw_gwphi", "cw_cos_inc", "cw_log10_mc",
    "cw_log10_fgw", "cw_log10_h", "cw_phase0", "cw_psi",
]

# Distance param names
DIST_DISCO_NAMES = [f"{psr.name}_cw_p_dist" for psr in disco_psrs]

# Noise param names (only used when SAMPLE_NOISE=True)
RN_DISCO_NAMES = []
for psr in disco_psrs:
    RN_DISCO_NAMES.append(f"{psr.name}_rednoise_log10_A")
    RN_DISCO_NAMES.append(f"{psr.name}_rednoise_gamma")
GWB_DISCO_NAMES = ["gwb_log10_A", "gwb_gamma"]

# Build the ordered list of sampled parameter names
if SAMPLE_NOISE:
    SAMPLED_NAMES = CW_DISCO_NAMES + DIST_DISCO_NAMES + RN_DISCO_NAMES + GWB_DISCO_NAMES
else:
    SAMPLED_NAMES = CW_DISCO_NAMES + DIST_DISCO_NAMES

Ndim = len(SAMPLED_NAMES)
N_CW = len(CW_DISCO_NAMES)       # 8
N_DIST = len(DIST_DISCO_NAMES)    # Npulsars

print(f"\nSampled parameters ({Ndim} total):")
for i, name in enumerate(SAMPLED_NAMES):
    print(f"  [{i:2d}] {name}")

# Fixed parameters: everything logl_disco needs that we're NOT sampling
FIXED_PARAMS = {}
for p in logl_disco.params:
    if p not in SAMPLED_NAMES:
        if p in disco_inj_params:
            FIXED_PARAMS[p] = disco_inj_params[p]
        else:
            # Try to find it in enterprise params
            found = False
            for ek, ev in enterprise_params.items():
                if ek == p:
                    FIXED_PARAMS[p] = ev
                    found = True
                    break
            if not found:
                print(f"  WARNING: parameter '{p}' required by logl but not in injection dict!")

print(f"\nFixed parameters ({len(FIXED_PARAMS)}):")
for k, v in sorted(FIXED_PARAMS.items()):
    print(f"  {k} = {v}")

In [ ]:
# =============================================================================
# PARAMETER BOUNDS

In [ ]:
# =============================================================================
CW_BOUNDS_LO = np.array([-1.0, 0.0,        -1.0, 7.0, -9.0, -18.0, 0.0,        0.0])
CW_BOUNDS_HI = np.array([ 1.0, 2*np.pi,     1.0, 10.0, -7.0, -11.0, 2*np.pi,   np.pi])
DIST_LO = 1e-6
DIST_HI = 30.0

if SAMPLE_NOISE:
    # Red noise: log10_A in [-18, -11], gamma in [0, 7]
    RN_BOUNDS_LO = np.tile([-18.0, 0.0], Npulsars)
    RN_BOUNDS_HI = np.tile([-11.0, 7.0], Npulsars)
    # GWB: same ranges
    GWB_BOUNDS_LO = np.array([-18.0, 0.0])
    GWB_BOUNDS_HI = np.array([-11.0, 7.0])
    PARAM_LO = np.concatenate([CW_BOUNDS_LO, [DIST_LO]*Npulsars, RN_BOUNDS_LO, GWB_BOUNDS_LO])
    PARAM_HI = np.concatenate([CW_BOUNDS_HI, [DIST_HI]*Npulsars, RN_BOUNDS_HI, GWB_BOUNDS_HI])
else:
    PARAM_LO = np.concatenate([CW_BOUNDS_LO, [DIST_LO]*Npulsars])
    PARAM_HI = np.concatenate([CW_BOUNDS_HI, [DIST_HI]*Npulsars])

In [ ]:
# =============================================================================
# LOG-POSTERIOR — wraps discovery's logl with flat-vector interface + priors

In [ ]:
# =============================================================================
def make_params_dict(x):
    """Convert flat parameter vector to discovery params dict."""
    d = dict(FIXED_PARAMS)  # start with fixed params
    for i, name in enumerate(SAMPLED_NAMES):
        d[name] = x[i]
    return d

@jax.jit
def logp(x):
    """Log-posterior: discovery's marginalized likelihood + distance prior.
    
    Uses PulsarLikelihood.logL which marginalizes over:
      - Timing model (improper prior GP)
      - Red noise GP coefficients (Woodbury identity)
      - GWB GP coefficients (Woodbury identity)
    
    The white noise covariance is fixed (efac=1, equad=1e-8).
    """
    # Hard prior: reject if any parameter is outside its prior bounds
    in_bounds = jnp.all(x >= jnp.array(PARAM_LO)) & jnp.all(x <= jnp.array(PARAM_HI))
    
    # Build params dict for discovery
    params = dict(FIXED_PARAMS)
    for i, name in enumerate(SAMPLED_NAMES):
        params[name] = x[i]
    
    # Woodbury-marginalized log-likelihood (marginalizes over timing model, red noise, and GWB GP coefficients)
    ll = logl_disco(params)
    
    # Gaussian EM distance prior: penalizes distances far from the EM measurement
    p_dists = x[N_CW:N_CW + N_DIST]
    log_prior = -0.5 * jnp.sum(jnp.square((p_dists - dist_mu_jnp) / sd_jnp))
    
    result = jnp.where(in_bounds, ll + log_prior, -1e30)
    # NaN guard: extreme parameter combos can overflow the Woodbury computation
    return jnp.where(jnp.isnan(result), -1e30, result)

In [ ]:
# =============================================================================
# TRUTH VECTOR

In [ ]:
# =============================================================================
truth_vals = []
for name in SAMPLED_NAMES:
    if name in disco_inj_params:
        truth_vals.append(disco_inj_params[name])
    elif name in enterprise_params:
        truth_vals.append(enterprise_params[name])
    else:
        raise ValueError(f"No injection value for '{name}'")
truth = np.array(truth_vals)

print("\nTruth vector:")
for i, (name, val) in enumerate(zip(SAMPLED_NAMES, truth)):
    print(f"  [{i:2d}] {name:40s} = {val:.6f}")

# Compile and verify
print("\nCompiling logp...")
lp_truth = float(logp(jnp.array(truth)))
print(f"logp(truth) = {lp_truth:.4f}")

In [ ]:
# =============================================================================
# DISTANCE FRINGE SPACING

In [ ]:
# =============================================================================
@jax.jit
def compute_delta_L(cos_gwtheta, gwphi, log10_fgw):
    """Distance fringe spacing [kpc] for each pulsar."""
    gwtheta = jnp.arccos(cos_gwtheta)
    f_gw    = 10.0 ** log10_fgw
    _, _, cos_mu = jax.vmap(lambda pos: fpcmu_fast(pos, gwtheta, gwphi))(psr_positions)
    denom = jnp.maximum(jnp.abs(1.0 - cos_mu), 1e-4)
    return 1.0 / (f_gw * KPC_OVER_C * denom)

dL = np.array(compute_delta_L(truth[0], truth[1], truth[4]))
print("\nDistance fringe spacings and modes per sigma:")
for i, psr in enumerate(disco_psrs):
    print(f"  {psr.name}: d_true={truth[N_CW+i]:.4f} kpc, dL={dL[i]:.6f} kpc, modes/sig={sd_arr[i]/dL[i]:.0f}")

In [ ]:
# =============================================================================
# HESSIAN / FISHER MATRIX

In [ ]:
# =============================================================================
grad_logp = jax.jit(jax.grad(logp))
batch_logp = jax.jit(jax.vmap(logp))

print("Computing Hessian at truth...")
t0 = time.time()
H_full = np.array(jax.hessian(logp)(jnp.array(truth)))
print(f"Hessian computed in {time.time()-t0:.1f}s")

# CW block eigendecomposition for Fisher proposal
H_cw = H_full[:N_CW, :N_CW]
eig_cw, evec_cw = np.linalg.eigh(-H_cw)
eig_cw_c = np.maximum(eig_cw, 1e-12 * eig_cw.max())

cov_fisher   = evec_cw @ np.diag(1.0 / eig_cw_c) @ evec_cw.T
cov_fisher   = 0.5 * (cov_fisher + cov_fisher.T)

eig_vals_cov, eig_vecs_cov = np.linalg.eigh(cov_fisher)
eig_sigs_fisher = 2.38 * np.sqrt(np.maximum(eig_vals_cov, 1e-30))

fisher_sig_cw = np.sqrt(np.diag(cov_fisher))

# Distance diagonal curvatures
H_dist_diag = np.array([H_full[N_CW+j, N_CW+j] for j in range(Npulsars)])

print(f"Fisher eigenmode widths: min={eig_sigs_fisher.min():.2e}, max={eig_sigs_fisher.max():.2e}")

# Pre-compile batch_logp
print("Pre-compiling batch_logp...")
_ = batch_logp(jnp.tile(jnp.array(truth), (40, 1)))
print("Done.")

In [ ]:
# =============================================================================
# STARTING POINT — offset 3-5 sigma from truth

In [ ]:
# =============================================================================
rng = np.random.default_rng(99)
x0  = truth.copy()

# Offset CW parameters by 3-5 Fisher sigmas in a random direction
# Cap the offset at 30% of the parameter range to avoid extreme bounds
for i in range(N_CW):
    offset_sigma = rng.uniform(3.0, 5.0) * rng.choice([-1, 1])
    proposed     = x0[i] + offset_sigma * fisher_sig_cw[i]
    lo = PARAM_LO[i] + 1e-4
    hi = PARAM_HI[i] - 1e-4
    # Clip to within 30% of parameter range from bounds (avoid extreme corners)
    margin = 0.3 * (hi - lo)
    x0[i] = np.clip(proposed, lo + margin, hi - margin)

for j in range(Npulsars):
    offset   = rng.uniform(3.0, 5.0) * rng.choice([-1, 1]) * sd_arr[j]
    x0[N_CW+j] = max(truth[N_CW+j] + offset, 0.01)

if SAMPLE_NOISE:
    # Offset noise params by small amounts from truth
    for j in range(N_CW + Npulsars, Ndim):
        x0[j] = truth[j] + rng.uniform(-0.5, 0.5)
        x0[j] = np.clip(x0[j], PARAM_LO[j] + 1e-4, PARAM_HI[j] - 1e-4)

lp_start = float(logp(jnp.array(x0)))
print(f"\nlogp(start) = {lp_start:.2f}")
print(f"logp(truth) = {lp_truth:.4f}")
print(f"Gap: {lp_truth - lp_start:.0f} nats")

In [ ]:
# =============================================================================
# SAMPLER UTILITIES

In [ ]:
# =============================================================================
def snap_distances(x_prop, n_newton=3):
    """
    Newton-snap pulsar distances to their conditional MAP given current CW params. Uses the gradient and diagonal Hessian curvature to take Newton steps. This is critical because distance has ~100s of fringe modes per EM sigma — without snapping, random distance proposals almost never land on a good fringe.
    """
    for _ in range(n_newton):
        g = np.array(grad_logp(jnp.array(x_prop)))
        for j in range(Npulsars):
            idx = N_CW + j
            if H_dist_diag[j] < -1e-6:
                x_prop[idx] = max(x_prop[idx] - g[idx] / H_dist_diag[j], 1e-6)
    return x_prop

def in_bounds(x_prop):
    """Check all parameters are within prior bounds."""
    return np.all(x_prop >= PARAM_LO) and np.all(x_prop <= PARAM_HI)

In [ ]:
# =============================================================================
# MCMC SAMPLER — 3-phase annealing

In [ ]:
# =============================================================================
# Exponential cooling schedule: T decreases geometrically each step
cool_rate = (T_end / T_start) ** (1.0 / n_anneal)
n_total   = n_anneal + n_adapt + n_prod

print(f"\nAnnealing: T {T_start} -> {T_end} over {n_anneal} steps (cool_rate={cool_rate:.6f})")
print(f"Adapt:     {n_adapt} steps at T=1 to build empirical covariance")
print(f"Prod:      {n_prod} steps")

all_chain = np.zeros((n_total, Ndim))
all_lps   = np.zeros(n_total)
all_temps = np.zeros(n_total)

x  = x0.copy()
lp = float(logp(jnp.array(x)))
T  = T_start

emp_samples = []
emp_cov     = None
use_emp_cov = False

scale_log = np.zeros(N_CW)

acc_counts = {'eigen': 0, 'dist': 0, 'joint': 0}
tot_counts = {'eigen': 0, 'dist': 0, 'joint': 0}

t0 = time.time()

# Main MCMC loop: 3 phases
#   1. Annealing (steps 0..n_anneal): temperature cools from T_start to 1.0
#      High T flattens the posterior, allowing broad exploration and mode-hopping.
#   2. Adapt (n_anneal..n_anneal+n_adapt): T=1, build empirical covariance from
#      the second half of annealing samples. This replaces the Fisher-based proposal
#      with one learned from the chain's actual shape.
#   3. Production (last n_prod steps): T=1, fixed proposals, samples saved for inference.
for step in range(n_total):
    if step < n_anneal:
        phase = 'anneal'
        T     = T_start * (cool_rate ** step)
    else:
        phase = 'adapt' if step < n_anneal + n_adapt else 'prod'
        T     = 1.0

    # At the annealing→adapt transition, build an empirical covariance matrix from the CW parameters of the late annealing samples. This adapts the proposal to the posterior's actual shape (which may differ from the Fisher approximation at truth).
    if step == n_anneal and len(emp_samples) > N_CW * 2:
        emp_arr = np.array(emp_samples)
        last_n  = max(N_CW * 3, len(emp_arr) // 3)
        emp_arr = emp_arr[-last_n:, :N_CW]

        if emp_arr.shape[0] > N_CW:
            raw_cov = np.cov(emp_arr.T)
            raw_cov = 0.5 * (raw_cov + raw_cov.T)
            eig_e, vec_e = np.linalg.eigh(raw_cov)
            eig_e = np.maximum(eig_e, 1e-12 * eig_e.max())
            emp_cov_mat = vec_e @ np.diag(eig_e) @ vec_e.T

            scale_nd = 2.38**2 / N_CW
            try:
                L_emp = np.linalg.cholesky(scale_nd * emp_cov_mat)
                emp_cov = {
                    'L':        L_emp,
                    'eig_sigs': 2.38 * np.sqrt(eig_e),
                    'vecs':     vec_e,
                }
                use_emp_cov = True
                print(f"  [step {step}] Switched to empirical covariance "
                      f"(built from {emp_arr.shape[0]} samples)")
            except np.linalg.LinAlgError:
                print(f"  [step {step}] Empirical Cholesky failed, keeping Fisher cov")

    # Choose proposal
    r = rng.random()

    if r < 0.50:
        # --- EIGENMODE PROPOSAL (50% of steps) ---
        # Proposes along a single eigenvector of the CW covariance matrix.
        # This is efficient because CW parameters are highly correlated (e.g., 
        # frequency-chirp mass degeneracy), so axis-aligned proposals would be very inefficient.
        # After proposing new CW params, Newton-snap the distances to their conditional MAP.
        if use_emp_cov:
            mode_idx = rng.integers(N_CW)
            z        = rng.standard_normal()
            sig      = emp_cov['eig_sigs'][mode_idx]
            x_prop   = x.copy()
            x_prop[:N_CW] += z * sig * emp_cov['vecs'][:, mode_idx]
        else:
            mode_idx   = rng.integers(N_CW)
            z          = rng.standard_normal()
            scaled_sig = eig_sigs_fisher[mode_idx] * np.sqrt(T) * np.exp(scale_log[mode_idx])
            x_prop     = x.copy()
            x_prop[:N_CW] += z * scaled_sig * eig_vecs_cov[:, mode_idx]

        x_prop = snap_distances(x_prop)

        if in_bounds(x_prop):
            lp_prop  = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            accepted  = np.log(rng.random() + 1e-300) < log_alpha
            if accepted:
                x = x_prop; lp = lp_prop
            acc_counts['eigen'] += int(accepted)
        tot_counts['eigen'] += 1

        if phase == 'anneal':
            # Robbins-Monro step-size adaptation during annealing:
            # adjusts eigenmode scales toward 35% acceptance rate
            gamma = 1.0 / (step + 100)
            scale_log[mode_idx] += gamma * (float(x is x_prop) - 0.35)
            scale_log[mode_idx]  = np.clip(scale_log[mode_idx], -5.0, 10.0)

    elif r < 0.80:
        # --- DISTANCE PROPOSAL (30% of steps) ---
        # Two-stage: (1) draw a new distance from the EM prior (broadened by sqrt(T)),
        # (2) grid-scan one fringe period around it to find the best fringe mode.
        # This handles the ~100s of fringe modes that exist within each pulsar's EM sigma.
        pi    = rng.integers(Npulsars)
        d_prop = rng.normal(mu_arr[pi], sd_arr[pi] * max(1.0, np.sqrt(T)))
        dL_j  = float(dL[pi])

        if d_prop > dL_j:
            x_snap = x.copy()
            x_snap[N_CW+pi] = d_prop

            d_lo    = max(d_prop - 0.6 * dL_j, 1e-6)
            d_hi    = d_prop + 0.6 * dL_j
            d_cands = np.linspace(d_lo, d_hi, 30)
            x_batch = np.tile(x_snap, (30, 1))
            x_batch[:, N_CW+pi] = d_cands
            lps_scan = np.array(batch_logp(jnp.array(x_batch)))

            x_prop = x.copy()
            x_prop[N_CW+pi] = float(d_cands[np.argmax(lps_scan)])
            lp_prop = float(logp(jnp.array(x_prop)))

            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['dist'] += 1
        tot_counts['dist'] += 1

    else:
        # --- JOINT CW PROPOSAL (20% of steps) ---
        # Proposes all CW parameters simultaneously from the full covariance (Fisher or empirical).
        # Low acceptance is expected in high dimensions but occasionally finds better modes.
        if use_emp_cov:
            z      = rng.standard_normal(N_CW)
            x_prop = x.copy()
            x_prop[:N_CW] += emp_cov['L'] @ z
        else:
            scale_nd = 2.38**2 / N_CW
            L_fish   = np.linalg.cholesky(scale_nd * T * cov_fisher)
            z        = rng.standard_normal(N_CW)
            x_prop   = x.copy()
            x_prop[:N_CW] += L_fish @ z

        if in_bounds(x_prop):
            lp_prop   = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['joint'] += 1
        tot_counts['joint'] += 1

    # Record state
    all_chain[step] = x
    all_lps[step]   = lp
    all_temps[step]  = T

    if phase == 'anneal' and step > n_anneal // 2:
        # Collect samples from the second half of annealing for empirical covariance estimation
        emp_samples.append(x.copy())

    if step % 2000 == 0:
        elapsed = time.time() - t0
        print(f"  step {step:5d}/{n_total} [{phase:6s}] T={T:7.2f} | logp={lp:10.2f} | "
              f"eigen={acc_counts['eigen']}/{tot_counts['eigen']} "
              f"dist={acc_counts['dist']}/{tot_counts['dist']} "
              f"joint={acc_counts['joint']}/{tot_counts['joint']} | {elapsed:.0f}s")

In [ ]:
# =============================================================================
# SUMMARY

In [ ]:
# =============================================================================
dt = time.time() - t0
print(f"\nDone in {dt:.1f}s ({n_total/dt:.0f} it/s)")
for k in acc_counts:
    ar = acc_counts[k] / max(tot_counts[k], 1)
    print(f"  {k:10s}: {acc_counts[k]:5d}/{tot_counts[k]:5d} = {ar:.3f}")

prod_chain = all_chain[n_anneal + n_adapt:]
prod_lps   = all_lps[n_anneal + n_adapt:]

print(f"\nlogp truth = {lp_truth:.4f}")
print(f"logp prod:  mean={np.mean(prod_lps):.2f}, std={np.std(prod_lps):.2f}, max={np.max(prod_lps):.2f}")

print("\nCW parameter recovery (production chain):")
for pidx, pname in enumerate(CW_DISCO_NAMES):
    med  = np.median(prod_chain[:, pidx])
    std  = np.std(prod_chain[:, pidx])
    bias = med - truth[pidx]
    print(f"  {pname:30s}: truth={truth[pidx]:+.4f}, median={med:+.4f}, std={std:.2e}, bias={bias:+.4f}")

print("\nDistance recovery (production chain):")
for j in range(Npulsars):
    med       = np.median(prod_chain[:, N_CW+j])
    err_modes = abs(med - truth[N_CW+j]) / dL[j]
    print(f"  {disco_psrs[j].name}: med={med:.4f}, truth={truth[N_CW+j]:.4f}, err={err_modes:.0f} fringe modes")

In [ ]:
# =============================================================================
# DIAGNOSTIC PLOTS

In [ ]:
# =============================================================================
ann_end   = n_anneal
adap_end  = n_anneal + n_adapt
steps_arr = np.arange(n_total)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
noise_label = "noise_vary" if SAMPLE_NOISE else "noise_fixed"
fig.suptitle(
    f'N_CW=1, h=10^{log10_h}, {noise_label}: PulsarLikelihood + Annealing\n'
    f'Start: dlogp={lp_start-lp_truth:.0f} from truth | {n_anneal}+{n_adapt}+{n_prod} steps',
    fontsize=13)

colour = '#1a3a5c'

def add_phase_lines(ax):
    ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.7)
    ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.7)

# Row 0, Col 0: logp trace
ax = axes[0, 0]
ax.plot(steps_arr, all_lps, color='#333', lw=0.3, alpha=0.8)
ax.axhline(lp_truth, color='r', ls='--', lw=1, label=f'truth ({lp_truth:.2f})')
add_phase_lines(ax)
ax.set_xlabel('step'); ax.set_ylabel('logp')
ax.set_title('Log-posterior trace'); ax.legend(fontsize=7)

# Row 0, Col 1: temperature schedule
ax = axes[0, 1]
ax.semilogy(steps_arr[:n_anneal], all_temps[:n_anneal], color='#b5442d', lw=0.6)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_xlabel('step'); ax.set_ylabel('Temperature T')
ax.set_title('Temperature schedule (annealing phase)')

# Row 0, Col 2: log10_fgw
ax = axes[0, 2]
ax.plot(steps_arr, all_chain[:, 4], color=colour, lw=0.3, alpha=0.8)
ax.axhline(truth[4], color='r', ls='--', lw=1)
ax.plot(0, x0[4], 'o', color=colour, ms=5, zorder=5)
add_phase_lines(ax)
ax.set_title('cw_log10_fgw'); ax.set_xlabel('step')

# Row 1: key CW parameters
for pidx, (param_col, param_name) in enumerate([(0, 'cw_cos_gwtheta'), (5, 'cw_log10_h'), (2, 'cw_cos_inc')]):
    ax = axes[1, pidx]
    ax.plot(steps_arr, all_chain[:, param_col], color=colour, lw=0.3, alpha=0.8)
    ax.axhline(truth[param_col], color='r', ls='--', lw=1)
    ax.plot(0, x0[param_col], 'o', color=colour, ms=5, zorder=5)
    add_phase_lines(ax)
    ax.set_title(param_name); ax.set_xlabel('step')

# Rows 2-3: distance traces and log10_h posterior
dist_slots = [(2, 0), (2, 1), (2, 2), (3, 0), (3, 1)]
for j in range(min(Npulsars, 5)):
    row, col = dist_slots[j]
    ax = axes[row, col]
    ax.plot(steps_arr, all_chain[:, N_CW+j], color=colour, lw=0.3, alpha=0.8)
    ax.axhline(truth[N_CW+j], color='r', ls='--', lw=1, label='truth')
    ax.plot(0, x0[N_CW+j], 'o', color='orange', ms=5, zorder=5, label='start')
    add_phase_lines(ax)
    ax.set_title(f'{disco_psrs[j].name} dist'); ax.set_xlabel('step')
    ax.legend(fontsize=7)

# Row 3, Col 2: log10_h posterior
ax = axes[3, 2]
ax.hist(prod_chain[:, 5], bins=50, color=colour, alpha=0.7)
ax.axvline(truth[5], color='r', ls='--', lw=1.5, label=f'truth={truth[5]}')
ax.set_title('cw_log10_h posterior (production)'); ax.set_xlabel('cw_log10_h'); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()